<a href="https://colab.research.google.com/github/GRUPO3MCDI500/GESTION-DE-DATOS-Y-TECNOLOGIAS/blob/main/notebooks/mcdi502_s2_g3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación Sumativa 2 — Taller I
## Análisis histórico de los Juegos Olímpicos con Apache Spark

**Asignatura:** MCDI502 — Gestión de Datos y Tecnologías  
**Grupo:** 3  
**Integrantes:**

- Eduardo Garrido
- Luis Espinosa
- Mauricio Ortega
- Wilson Arévalo

Este Notebook implementa un flujo reproducible con **RDDs, DataFrames y Spark SQL**. El desarrollo sigue la pauta de la evaluación: configuración del entorno, creación y transformación de RDDs, integración de las cinco entidades olímpicas, optimización, particionamiento, columnas calculadas y agregaciones.

> Ejecute las celdas en orden. El Notebook acepta los nombres originales del repositorio y las variantes singular/plural mencionadas en la pauta.

In [1]:
from google.colab import files

archivos_subidos = files.upload()

Saving GESTION-DE-DATOS-Y-TECNOLOGIAS-adaptado.zip to GESTION-DE-DATOS-Y-TECNOLOGIAS-adaptado.zip


In [2]:
import os

ZIP_PATH = "/content/GESTION-DE-DATOS-Y-TECNOLOGIAS-adaptado.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"No se encontró el archivo: {ZIP_PATH}")

!unzip -o -q "$ZIP_PATH" -d /content

print("Proyecto descomprimido correctamente.")

Proyecto descomprimido correctamente.


In [3]:
from pathlib import Path

CARPETA_DATOS = Path("/content/data/raw")

archivos_requeridos = [
    "deportista.csv",
    "deportista2.csv",
    "evento.csv",
    "equipo.csv",
    "resultados.csv",
    "juegos.json",
]

print("Verificación de archivos:\n")

faltantes = []

for nombre in archivos_requeridos:
    ruta = CARPETA_DATOS / nombre
    estado = "OK" if ruta.exists() else "FALTANTE"
    print(f"{estado:8} {ruta}")

    if not ruta.exists():
        faltantes.append(nombre)

if faltantes:
    raise FileNotFoundError(
        f"Faltan los siguientes archivos: {faltantes}"
    )

print("\nTodos los archivos están disponibles.")

Verificación de archivos:

OK       /content/data/raw/deportista.csv
OK       /content/data/raw/deportista2.csv
OK       /content/data/raw/evento.csv
OK       /content/data/raw/equipo.csv
OK       /content/data/raw/resultados.csv
OK       /content/data/raw/juegos.json

Todos los archivos están disponibles.


## Correspondencia con la pauta

| Sección | Evidencia implementada |
|---|---|
| 1 | Java, PySpark, variables de entorno, `SparkSession` y `SparkContext` |
| 2 | RDD `deportista` con 6 particiones, `deportista2`, unión y conversión a DataFrame |
| 3 | Mayores de edad, mujeres y transformación a mayúsculas |
| 4 | DataFrames Evento, Resultado, Equipo y Juego; caché, persistencia y joins |
| 5 | DataFrame integrado reparticionado a 5 particiones |
| 6 | Cantidad de filas, tipos de datos y esquema |
| 7 | IMC y `Descripción_sexo` |
| 8 | Cuatro agregaciones mediante Spark SQL |

# 1. Configuración inicial del entorno

Se utiliza Java 17 y PySpark 4.2.0. Además, se habilita Adaptive Query Execution para que Spark pueda optimizar dinámicamente las operaciones de intercambio y las particiones de *shuffle*.

In [ ]:
# Instalación orientada a Google Colab.
# En una ejecución local con las dependencias instaladas, esta celda puede omitirse.
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q pyspark==4.2.0

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import csv
import os
from pathlib import Path

import pyspark
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)

spark = (
    SparkSession.builder
    .appName("MCDI502_S2_Grupo3_Olimpiadas")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Aplicación:", sc.appName)
print("Versión de Spark:", spark.version)
print("Paralelismo predeterminado:", sc.defaultParallelism)

## Localización y validación de archivos

El Notebook busca los datos en rutas habituales de Colab y del repositorio. Se aceptan las variantes `evento/eventos`, `equipo/equipos`, `resultado/resultados` y `juego/juegos`.

In [ ]:
ALIAS_ARCHIVOS = {
    "deportista": ["deportista.csv"],
    "deportista2": ["deportista2.csv"],
    "evento": ["evento.csv", "eventos.csv"],
    "equipo": ["equipo.csv", "equipos.csv"],
    "resultado": ["resultados.csv", "resultado.csv"],
    "juego": ["juegos.json", "juego.json"],
}

RUTAS_CANDIDATAS = [
    Path("data/raw"),
    Path("../data/raw"),
    Path("/content/data/raw"),
    Path("/content"),
    Path.cwd(),
]


def resolver_archivo(alias):
    for carpeta in RUTAS_CANDIDATAS:
        for nombre in ALIAS_ARCHIVOS[alias]:
            ruta = (carpeta / nombre).resolve()
            if ruta.exists():
                return ruta
    return None


RUTAS = {alias: resolver_archivo(alias) for alias in ALIAS_ARCHIVOS}
faltantes = [alias for alias, ruta in RUTAS.items() if ruta is None]

if faltantes:
    detalle = {alias: ALIAS_ARCHIVOS[alias] for alias in faltantes}
    raise FileNotFoundError(
        "No se encontraron todos los archivos requeridos. "
        f"Faltantes: {detalle}. Cargue los archivos en /content o data/raw."
    )

for alias, ruta in RUTAS.items():
    print(f"{alias:12s} -> {ruta}")

# 2. RDDs: creación y unión (ID 3.1)

Los archivos de deportistas no poseen encabezado. Se utiliza `csv.reader` para respetar el formato y se corrige una fila del archivo original que contiene una coma vacía al final. Los valores `0` se conservan en esta etapa porque forman parte de los datos originales y serán tratados como faltantes en los cálculos estadísticos.

In [ ]:
def parsear_deportista(linea):
    valores = next(csv.reader([linea]))

    # El archivo original contiene una fila con una columna vacía adicional al final.
    if len(valores) == 8 and valores[-1].strip() == "":
        valores = valores[:-1]

    if len(valores) != 7:
        raise ValueError(f"Fila de deportista inválida ({len(valores)} columnas): {linea}")

    return (
        int(valores[0]),       # deportista_id
        valores[1].strip(),    # nombre
        int(valores[2]),       # genero: 1 hombre, 2 mujer
        int(float(valores[3])),# edad
        float(valores[4]),     # altura en centímetros
        float(valores[5]),     # peso en kilogramos
        int(valores[6]),       # equipo_id
    )


# La pauta solicita un RDD llamado deportista con 6 particiones.
deportista = (
    sc.textFile(str(RUTAS["deportista"]), minPartitions=6)
    .map(parsear_deportista)
    .repartition(6)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Segundo archivo de deportistas.
deportista2 = (
    sc.textFile(str(RUTAS["deportista2"]), minPartitions=6)
    .map(parsear_deportista)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Unión solicitada.
deportistaTotal = deportista.union(deportista2).persist(StorageLevel.MEMORY_AND_DISK)

cantidad_deportista_total = deportistaTotal.count()

print("Particiones de deportista:", deportista.getNumPartitions())
print("Particiones de deportista2:", deportista2.getNumPartitions())
print("Registros de deportistaTotal:", cantidad_deportista_total)
print("Primeros registros:")
for fila in deportistaTotal.take(5):
    print(fila)

In [ ]:
esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("genero", IntegerType(), False),
    StructField("edad", IntegerType(), False),
    StructField("altura", DoubleType(), False),
    StructField("peso", DoubleType(), False),
    StructField("equipo_id", IntegerType(), False),
])

# Se conserva una referencia al RDD antes de utilizar el nombre solicitado para el DataFrame.
deportista_rdd = deportista

deportista = (
    spark.createDataFrame(deportistaTotal, esquema_deportista)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Materialización de la persistencia.
print("Registros del DataFrame deportista:", deportista.count())
deportista.show(10, truncate=False)
deportista.printSchema()

# 3. RDDs: transformaciones (ID 3.2)

Se aplican transformaciones *lazy* sobre `deportistaTotal`. Se considera mayor de edad a quien tiene **18 años o más**. Para la conversión a mayúsculas se transforman todos los valores textuales de cada registro; en este modelo, el único campo textual es el nombre.

In [ ]:
MayorEdad = deportistaTotal.filter(lambda fila: fila[3] >= 18)

Deportistas_mujer = deportistaTotal.filter(lambda fila: fila[2] == 2)

deportistaTotal_mayusculas = deportistaTotal.map(
    lambda fila: tuple(
        valor.upper() if isinstance(valor, str) else valor
        for valor in fila
    )
)

print("Cantidad de deportistas mayores de edad:", MayorEdad.count())
print("Cantidad de deportistas mujeres:", Deportistas_mujer.count())
print("Muestra en mayúsculas:")
for fila in deportistaTotal_mayusculas.take(5):
    print(fila)

# 4. DataFrames: creación, integración y optimización (ID 3.1 y 3.4)

- `Equipo` se lee desde CSV con esquema explícito.
- `Evento` requiere una función de limpieza porque algunas filas del archivo están completamente entrecomilladas cuando el nombre del evento contiene comas.
- `Resultado` utiliza punto y coma como separador y convierte `#N/A` en nulo.
- `Juego` se lee desde JSON. En el archivo original, el año está almacenado en `temporada` y la estación olímpica en `ciudad`; se normalizan como `anio` y `temporada` para que el análisis sea coherente.

In [ ]:
esquema_equipo_original = StructType([
    StructField("id", IntegerType(), False),
    StructField("equipo", StringType(), False),
    StructField("sigla", StringType(), True),
])

equipo = (
    spark.read
    .option("header", True)
    .schema(esquema_equipo_original)
    .csv(str(RUTAS["equipo"]))
    .withColumnRenamed("id", "equipo_id")
)


def parsear_evento(linea):
    texto = linea.strip()
    if texto.startswith('"') and texto.endswith('"'):
        texto = texto[1:-1].replace('""', '"')

    valores = next(csv.reader([texto]))
    if len(valores) != 3:
        raise ValueError(f"Fila de evento inválida: {linea}")

    deporte_id = None if valores[2].strip() == "#N/A" else int(valores[2])
    return int(valores[0]), valores[1].strip(), deporte_id


lineas_evento = sc.textFile(str(RUTAS["evento"]))
encabezado_evento = lineas_evento.first()

evento_rdd = (
    lineas_evento
    .filter(lambda linea: linea != encabezado_evento)
    .map(parsear_evento)
)

esquema_evento = StructType([
    StructField("evento_id", IntegerType(), False),
    StructField("evento", StringType(), False),
    StructField("deporte_id", IntegerType(), True),
])

evento = spark.createDataFrame(evento_rdd, esquema_evento)

esquema_resultado = StructType([
    StructField("resultado_id", IntegerType(), False),
    StructField("medalla", StringType(), True),
    StructField("deportista_id", IntegerType(), False),
    StructField("juego_id", IntegerType(), False),
    StructField("evento_id", IntegerType(), True),
])

resultado = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("nullValue", "#N/A")
    .option("mode", "PERMISSIVE")
    .schema(esquema_resultado)
    .csv(str(RUTAS["resultado"]))
)

esquema_juego_original = StructType([
    StructField("juego_id", IntegerType(), False),
    StructField("ano", StringType(), False),
    StructField("temporada", IntegerType(), False),
    StructField("ciudad", StringType(), False),
])

juego = (
    spark.read
    .option("multiline", True)
    .schema(esquema_juego_original)
    .json(str(RUTAS["juego"]))
    .select(
        "juego_id",
        F.col("ano").alias("edicion"),
        F.col("temporada").alias("anio"),
        F.col("ciudad").alias("temporada"),
    )
)

for nombre, df in [
    ("deportista", deportista),
    ("evento", evento),
    ("equipo", equipo),
    ("resultado", resultado),
    ("juego", juego),
]:
    print()
    print(nombre.upper())
    df.show(5, truncate=False)
    df.printSchema()

In [ ]:
# Optimización: persistencia de las tablas, materialización y uso posterior de broadcast joins.
# Resultado y deportista son las tablas de mayor tamaño, por lo que se usa MEMORY_AND_DISK.
deportista.persist(StorageLevel.MEMORY_AND_DISK)
resultado.persist(StorageLevel.MEMORY_AND_DISK)

# Las dimensiones pequeñas se mantienen en memoria.
evento.cache()
equipo.cache()
juego.cache()

for nombre, df in [
    ("deportista", deportista),
    ("evento", evento),
    ("equipo", equipo),
    ("resultado", resultado),
    ("juego", juego),
]:
    print(
        f"{nombre:12s}: {df.count():>7} registros | "
        f"persistencia = {df.storageLevel}"
    )

print(
    "Resultados con evento_id nulo:",
    resultado.filter(F.col("evento_id").isNull()).count(),
)

In [ ]:
# Resultado es la tabla central del modelo y conecta deportistas, equipos, juegos y eventos.
# Las dimensiones pequeñas se transmiten mediante broadcast para evitar shuffles innecesarios.
datos_olimpicos = (
    resultado
    .join(deportista, on="deportista_id", how="inner")
    .join(F.broadcast(equipo), on="equipo_id", how="left")
    .join(F.broadcast(evento), on="evento_id", how="left")
    .join(F.broadcast(juego), on="juego_id", how="left")
    .withColumn("equipo", F.coalesce(F.col("equipo"), F.lit("Equipo no informado")))
    .withColumn("evento", F.coalesce(F.col("evento"), F.lit("Evento no informado")))
)

cantidad_integrada = datos_olimpicos.count()
print("Cantidad de filas del DataFrame integrado:", cantidad_integrada)
datos_olimpicos.show(10, truncate=False)

# 5. Paralelismo (ID 3.4)

El DataFrame integrado se reparte explícitamente en cinco particiones. Se utiliza `equipo_id` como clave para distribuir conjuntamente registros del mismo equipo durante análisis posteriores.

In [ ]:
datos_olimpicos = (
    datos_olimpicos
    .repartition(5, "equipo_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Se materializa la nueva distribución.
datos_olimpicos.count()

numero_particiones = datos_olimpicos.rdd.getNumPartitions()
print("Número de particiones:", numero_particiones)
assert numero_particiones == 5, "El DataFrame debe tener exactamente 5 particiones."

# 6. Inspección del DataFrame (ID 3.1)

In [ ]:
print("Cantidad de filas:", datos_olimpicos.count())
print("Tipos de datos:")
for nombre, tipo in datos_olimpicos.dtypes:
    print(f"- {nombre}: {tipo}")

print()
print("Esquema completo:")
datos_olimpicos.printSchema()

# 7. Columnas calculadas

Los valores `0` en edad, altura o peso representan datos ausentes. Por ello, el IMC se calcula únicamente cuando altura y peso son mayores que cero. También se crean columnas auxiliares con valores válidos para evitar que los ceros distorsionen las estadísticas.

In [ ]:
datos_olimpicos = (
    datos_olimpicos
    .withColumn(
        "IMC",
        F.when(
            (F.col("altura") > 0) & (F.col("peso") > 0),
            F.round(F.col("peso") / F.pow(F.col("altura") / F.lit(100.0), 2), 2),
        ).otherwise(F.lit(None).cast(DoubleType())),
    )
    .withColumn(
        "Descripción_sexo",
        F.when(F.col("genero") == 1, F.lit("Hombre"))
        .when(F.col("genero") == 2, F.lit("Mujer"))
        .otherwise(F.lit("No informado")),
    )
    .withColumn(
        "medalla_descripcion",
        F.when(F.col("medalla") == "Gold", F.lit("Oro"))
        .when(F.col("medalla") == "Silver", F.lit("Plata"))
        .when(F.col("medalla") == "Bronze", F.lit("Bronce"))
        .otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "edad_valida",
        F.when(F.col("edad") > 0, F.col("edad")).otherwise(F.lit(None).cast(IntegerType())),
    )
    .withColumn(
        "altura_valida",
        F.when(F.col("altura") > 0, F.col("altura")).otherwise(F.lit(None).cast(DoubleType())),
    )
)

datos_olimpicos.select(
    "deportista_id",
    "nombre",
    "genero",
    "Descripción_sexo",
    "edad",
    "altura",
    "peso",
    "IMC",
).show(15, truncate=False)

# 8. Agregaciones requeridas mediante Spark SQL (ID 3.3)

El DataFrame se registra como vista temporal. Las cuatro consultas se desarrollan con Spark SQL para demostrar manipulación y exploración de datos estructurados.

In [ ]:
datos_olimpicos.createOrReplaceTempView("olimpicos")

## 8.1 Cantidad de medallas por equipo

In [ ]:
medallas_por_equipo = spark.sql("""
    SELECT
        equipo,
        SUM(CASE WHEN medalla = 'Gold' THEN 1 ELSE 0 END) AS Oro,
        SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS Plata,
        SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS Bronce,
        SUM(CASE WHEN medalla IN ('Gold', 'Silver', 'Bronze') THEN 1 ELSE 0 END)
            AS Total_medallas
    FROM olimpicos
    GROUP BY equipo
    HAVING Total_medallas > 0
    ORDER BY Total_medallas DESC, equipo ASC
""")

medallas_por_equipo.show(30, truncate=False)

## 8.2 Suma, promedio, máximo y mínimo de edad por tipo de medalla

In [ ]:
estadisticas_edad_medalla = spark.sql("""
    SELECT
        medalla_descripcion AS tipo_medalla,
        SUM(edad_valida) AS suma_edad,
        ROUND(AVG(edad_valida), 2) AS promedio_edad,
        MAX(edad_valida) AS maximo_edad,
        MIN(edad_valida) AS minimo_edad
    FROM olimpicos
    WHERE medalla_descripcion IS NOT NULL
    GROUP BY medalla_descripcion
    ORDER BY tipo_medalla
""")

estadisticas_edad_medalla.show(truncate=False)

## 8.3 DataFrame `temporada`: estadísticas de altura por temporada

In [ ]:
temporada = spark.sql("""
    SELECT
        temporada,
        ROUND(SUM(altura_valida), 2) AS suma_altura,
        ROUND(AVG(altura_valida), 2) AS promedio_altura,
        MAX(altura_valida) AS maximo_altura,
        MIN(altura_valida) AS minimo_altura
    FROM olimpicos
    WHERE temporada IS NOT NULL
    GROUP BY temporada
    ORDER BY temporada
""")

temporada.show(truncate=False)

## 8.4 DataFrame `sexo`: estadísticas de edad por sexo

In [ ]:
sexo = spark.sql("""
    SELECT
        `Descripción_sexo` AS sexo,
        SUM(edad_valida) AS suma_edad,
        ROUND(AVG(edad_valida), 2) AS promedio_edad,
        MAX(edad_valida) AS maximo_edad,
        MIN(edad_valida) AS minimo_edad
    FROM olimpicos
    GROUP BY `Descripción_sexo`
    ORDER BY sexo
""")

sexo.show(truncate=False)

## Revisión del plan de ejecución

`explain` permite comprobar que Spark aplica agregaciones distribuidas, intercambios y las optimizaciones habilitadas. Los `BroadcastHashJoin` utilizados durante la integración reducen el costo de unir las dimensiones pequeñas.

In [ ]:
medallas_por_equipo.explain(mode="formatted")

# 9. Validaciones finales y conclusiones

Las siguientes comprobaciones verifican los nombres de columnas, el número de particiones y la existencia de resultados para cada consulta requerida.

In [ ]:
columnas_obligatorias = {
    "deportista_id",
    "nombre",
    "genero",
    "edad",
    "altura",
    "peso",
    "equipo_id",
    "evento_id",
    "juego_id",
    "medalla",
    "IMC",
    "Descripción_sexo",
}

faltan_columnas = columnas_obligatorias.difference(datos_olimpicos.columns)
assert not faltan_columnas, f"Faltan columnas obligatorias: {faltan_columnas}"
assert datos_olimpicos.rdd.getNumPartitions() == 5
assert medallas_por_equipo.count() > 0
assert estadisticas_edad_medalla.count() == 3
assert temporada.count() >= 2
assert sexo.count() >= 2

print("Validaciones completadas correctamente.")
print("Filas integradas:", datos_olimpicos.count())
print("Equipos con medallas:", medallas_por_equipo.count())

## Conclusiones técnicas

1. Los RDDs permiten controlar la lectura y limpieza inicial de datos semiestructurados, incluyendo filas irregulares.
2. Los DataFrames facilitan la aplicación de esquemas, la integración mediante claves y la optimización del plan de ejecución.
3. La persistencia evita recalcular fuentes reutilizadas, mientras que los *broadcast joins* reducen movimientos de datos al unir dimensiones pequeñas.
4. El reparticionamiento en cinco particiones cumple el requisito de paralelismo y deja explícita la distribución del DataFrame final.
5. Spark SQL permite expresar de forma clara y auditable las agregaciones requeridas por equipo, medalla, temporada y sexo.

In [ ]:
# Liberación opcional de recursos al finalizar la revisión.
for df in [deportista, evento, equipo, resultado, juego, datos_olimpicos]:
    df.unpersist(blocking=False)

spark.catalog.clearCache()
print("Recursos liberados.")